# ChiliEdgeNet — Cross-dataset evaluation (Chili Leaf Disease Dataset, Bangladesh)

Danh gia model de xuat (da train tren ChiliEdge-DS, **khong train lai**) tren mot
dataset benh ot **doc lap** — khac nguoi thu thap, khac may anh, khac dia diem
(Bangladesh) — de kiem tra kha nang tong quat hoa vuot ra ngoai du lieu tu thu thap.

**Nguon:** "Chili Leaf Disease Dataset: Annotated Smartphone Images of Anthracnose,
Cercospora Leaf Spot, Leaf Curl Disease, and Healthy Leaves in Bangladesh" — Mendeley
Data, DOI `wzc6r6w5w5` v2, license **CC BY 4.0**. 1.544 anh, 4 lop:
`Anthracnose`(351), `Cercospora Leaf Spot`(371), `Leaf Curl Disease`(383),
`Healthy Leaves`(439). Anh nen trang (giong dieu kien thu thap ChiliEdge-DS).

**Anh xa nhan** (4/5 lop khop truc tiep; thieu `yellow_leaf` — dataset ngoai khong
co lop tuong duong nen khong danh gia rieng duoc lop nay):

| Ten lop trong dataset ngoai | Anh xa sang lop cua minh |
|---|---|
| Anthracnose | anthracnose |
| Healthy Leaves | healthy |
| Leaf Curl Disease | leaf_curl |
| Cercospora Leaf Spot | leaf_spot |

**Input can Add tren Kaggle:**
1. Checkpoint `c14_s2*.pth` (dataset `model-ot` da dung o cac NB khac).
2. Bo Chili Leaf Disease Dataset (Bangladesh) — neu chua co tren Kaggle, tai tu
   Mendeley (`data.mendeley.com/datasets/wzc6r6w5w5/2`) roi upload thanh 1 Kaggle
   Dataset rieng, giu nguyen ten thu muc theo lop.

In [ ]:
import os, glob, csv, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ===== CONFIG - SUA LAI 2 duong dan nay cho dung voi Input da Add tren Kaggle =====

CKPT_DIR = '/kaggle/input/datasets/tminhhi/model-ot'
CKPT_NAME_CANDIDATES = ['c14_s2.pth', 'c14_s2_seed123.pth']

# Thu muc GOC cua Chili Leaf Disease Dataset (Bangladesh) sau khi Add Input.
CROSS_ROOT = '/kaggle/input/chili-leaf-disease-dataset-bangladesh'

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASS_NAMES = ['anthracnose', 'healthy', 'leaf_curl', 'leaf_spot', 'yellow_leaf']

# Anh xa ten thu muc (dataset ngoai) -> ten lop cua minh. Ten thu muc thuc te co the
# khac chut it (hoa/thuong, dau cach/gach duoi) - them bien the vao day neu can.
LABEL_MAP = {
    'anthracnose': 'anthracnose',
    'cercospora leaf spot': 'leaf_spot',
    'cercospora_leaf_spot': 'leaf_spot',
    'leaf curl disease': 'leaf_curl',
    'leaf_curl_disease': 'leaf_curl',
    'leaf curl': 'leaf_curl',
    'healthy leaves': 'healthy',
    'healthy_leaves': 'healthy',
    'healthy': 'healthy',
}

TF_EMBED_DIM, TF_DEPTH = 128, 1
TF_HEADS, TF_KV_HEADS = 4, 2
TF_MLP_RATIO, TF_DROP, TF_ROPE_BASE = 2.0, 0.1, 100.0
BACKBONE_CUT = 14

In [ ]:
# ===== Kien truc ChiliEdgeNet - COPY Y HET model.py / NB1 de dam bao load dung checkpoint =====
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_axial_rope(gh, gw, head_dim, base=100.0):
    d = head_dim // 2
    inv = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    gy = torch.arange(gh).float().view(gh, 1).expand(gh, gw).reshape(-1)
    gx = torch.arange(gw).float().view(1, gw).expand(gh, gw).reshape(-1)
    ey = torch.cat([torch.outer(gy, inv)] * 2, dim=-1)
    ex = torch.cat([torch.outer(gx, inv)] * 2, dim=-1)
    return ey.cos()[None,None], ey.sin()[None,None], ex.cos()[None,None], ex.sin()[None,None]

def _rot_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope2d(t, cy, sy, cx, sx):
    d = t.shape[-1] // 2
    ty, tx = t[..., :d], t[..., d:]
    ty = ty * cy + _rot_half(ty) * sy
    tx = tx * cx + _rot_half(tx) * sx
    return torch.cat([ty, tx], dim=-1)

class GroupedQueryAttention(nn.Module):
    def __init__(self, dim, nh, nkv, dropout=0.0):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, dim // nh
        self.groups, self.scale = nh // nkv, (dim // nh) ** -0.5
        self.q = nn.Linear(dim, nh * self.hd, bias=False)
        self.k = nn.Linear(dim, nkv * self.hd, bias=False)
        self.v = nn.Linear(dim, nkv * self.hd, bias=False)
        self.o = nn.Linear(nh * self.hd, dim, bias=False)
        self.ad, self.pd = nn.Dropout(dropout), nn.Dropout(dropout)
    def forward(self, x, rope):
        B, N, _ = x.shape; cy, sy, cx, sx = rope
        q = self.q(x).view(B, N, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, N, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, N, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope2d(q, cy, sy, cx, sx), apply_rope2d(k, cy, sy, cx, sx)
        if self.groups > 1:
            k = k.repeat_interleave(self.groups, dim=1)
            v = v.repeat_interleave(self.groups, dim=1)
        attn = self.ad(((q @ k.transpose(-2, -1)) * self.scale).softmax(-1))
        return self.pd(self.o((attn @ v).transpose(1, 2).reshape(B, N, -1)))

class TinyLLMBlock(nn.Module):
    def __init__(self, dim, nh, nkv, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.n1 = RMSNorm(dim); self.attn = GroupedQueryAttention(dim, nh, nkv, dropout)
        self.n2 = RMSNorm(dim); h = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(h, dim), nn.Dropout(dropout))
    def forward(self, x, rope):
        x = x + self.attn(self.n1(x), rope)
        return x + self.mlp(self.n2(x))

class ChiliEdgeNet(nn.Module):
    def __init__(self, num_classes, pretrained=False, cut=BACKBONE_CUT,
                 embed_dim=TF_EMBED_DIM, depth=TF_DEPTH, heads=TF_HEADS, kv_heads=TF_KV_HEADS,
                 mlp_ratio=TF_MLP_RATIO, drop=TF_DROP, rope_base=TF_ROPE_BASE):
        super().__init__()
        w = torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        base = torchvision.models.mobilenet_v2(weights=w)
        self.first_conv = base.features[0]
        self.backbone = nn.Sequential(*list(base.features[1:cut]))
        with torch.no_grad():
            feat = self.backbone(self.first_conv(torch.zeros(1,3,IMG_SIZE,IMG_SIZE)))
        c_out, grid = feat.shape[1], feat.shape[2]
        self.reduce = nn.Sequential(nn.Conv2d(c_out, embed_dim, 1, bias=False),
                                    nn.BatchNorm2d(embed_dim), nn.GELU())
        self.emb_drop = nn.Dropout(drop)
        cy, sy, cx, sx = build_axial_rope(grid, grid, embed_dim // heads, base=rope_base)
        for n, b in zip(['rcy','rsy','rcx','rsx'], [cy, sy, cx, sx]):
            self.register_buffer(n, b, persistent=False)
        self.encoder = nn.ModuleList([TinyLLMBlock(embed_dim, heads, kv_heads, mlp_ratio, drop)
                                      for _ in range(depth)])
        self.norm = RMSNorm(embed_dim)
        self.drop = nn.Dropout(0.2)
        self.fc = nn.Linear(embed_dim, num_classes)
        print(f'[ChiliEdgeNet] backbone out: {c_out}ch @ {grid}x{grid} -> {grid*grid} token')
    def forward(self, x):
        x = self.backbone(self.first_conv(x))
        x = self.emb_drop(self.reduce(x).flatten(2).transpose(1, 2))
        rope = (self.rcy, self.rsy, self.rcx, self.rsx)
        for blk in self.encoder:
            x = blk(x, rope)
        return self.fc(self.drop(self.norm(x).mean(dim=1)))

print('ChiliEdgeNet loaded.')

In [ ]:
# ===== Nap checkpoint (tim tu dong trong /kaggle/input) =====
def _find_ckpt():
    for name in CKPT_NAME_CANDIDATES:
        cands = ([os.path.join(CKPT_DIR, name)] +
                 sorted(glob.glob('/kaggle/input/**/' + name, recursive=True)))
        for p in cands:
            if os.path.exists(p):
                return p
    return None

CKPT_PATH = _find_ckpt()
assert CKPT_PATH, f'Khong tim thay checkpoint (thu: {CKPT_NAME_CANDIDATES}) - kiem tra da Add Input chua'
print('Checkpoint:', CKPT_PATH)

model = ChiliEdgeNet(len(CLASS_NAMES), pretrained=False).to(DEVICE)
state = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(state, strict=True)
model.eval()
print('Da nap model.')

In [ ]:
# ===== Tim dataset Bangladesh + gom danh sach anh + nhan da anh xa =====
def _find_cross_root():
    cands = ([CROSS_ROOT] + glob.glob('/kaggle/input/*chili*disease*') +
             glob.glob('/kaggle/input/*chilli*disease*') + glob.glob('/kaggle/input/*wzc6r6w5w5*'))
    for c in cands:
        if not os.path.isdir(c):
            continue
        for root, dirs, files in os.walk(c):
            norm_dirs = [d.lower().replace('_', ' ') for d in dirs]
            if any(d in LABEL_MAP for d in norm_dirs):
                return root
    return None

CROSS_DIR = _find_cross_root()
assert CROSS_DIR, ('Khong tu tim thay thu muc dataset Bangladesh - kiem tra da Add Input chua, '
                    'va sua CROSS_ROOT o cell CONFIG cho dung duong dan /kaggle/input/...')
print('Cross-dataset dir:', CROSS_DIR)
print('Thu muc con:', os.listdir(CROSS_DIR))

cross_items = []
for folder in os.listdir(CROSS_DIR):
    fdir = os.path.join(CROSS_DIR, folder)
    if not os.path.isdir(fdir):
        continue
    mapped = LABEL_MAP.get(folder.lower().replace('_', ' '))
    if mapped is None:
        print(f'[!] Bo qua thu muc khong nhan dien duoc nhan: {folder}')
        continue
    for fn in sorted(os.listdir(fdir)):
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            cross_items.append((os.path.join(fdir, fn), folder, mapped))
print(f'{len(cross_items)} anh se chay qua model, {len(set(m for _,_,m in cross_items))} lop khop.')

In [ ]:
# ===== Chay suy luan tren toan bo anh Bangladesh =====
tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

@torch.no_grad()
def predict(img):
    x = tf(img).unsqueeze(0).to(DEVICE)
    probs = torch.softmax(model(x), dim=1)[0]
    conf, idx = probs.max(dim=0)
    return CLASS_NAMES[idx.item()], conf.item() * 100

cross_rows = []
t0 = time.perf_counter()
for i, (path, orig_folder, true_label) in enumerate(cross_items):
    img = Image.open(path).convert('RGB')
    pred_class, conf = predict(img)
    cross_rows.append({'image_path': path, 'source_folder': orig_folder, 'true_label': true_label,
                        'predicted_class': pred_class, 'confidence': conf, 'correct': pred_class == true_label})
    if (i + 1) % 200 == 0:
        print(f'  ...{i+1}/{len(cross_items)} anh ({time.perf_counter()-t0:.1f}s)')

cross_df = pd.DataFrame(cross_rows)
cross_df.to_csv('cross_dataset_bangladesh_predictions.csv', index=False)
print('Xong. Da luu cross_dataset_bangladesh_predictions.csv')

In [ ]:
# ===== Tong hop Cross-dataset evaluation =====
n = len(cross_rows)
correct = sum(1 for r in cross_rows if r['correct'])
acc = correct / n * 100
confs = np.array([r['confidence'] for r in cross_rows])

y_true = [r['true_label'] for r in cross_rows]
y_pred = [r['predicted_class'] for r in cross_rows]
matched_labels = sorted(set(y_true))

print('Nguon: Chili Leaf Disease Dataset (Bangladesh, Mendeley wzc6r6w5w5)')
print(f'Tong so anh danh gia: {n} (tren {len(matched_labels)}/5 lop khop: {matched_labels})')
print()
print(f'CROSS-DATASET ACCURACY: {acc:.2f}% ({correct}/{n})')
print(f'Confidence top-1 trung binh: {confs.mean():.2f}% (median {np.median(confs):.2f}%)')
print()
print(classification_report(y_true, y_pred, labels=matched_labels, target_names=matched_labels, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=matched_labels)
print('Confusion matrix (hang=that, cot=du doan), thu tu', matched_labels, ':')
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=matched_labels, yticklabels=matched_labels, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True (Bangladesh dataset)')
ax.set_title('Cross-dataset evaluation - Confusion Matrix')
plt.tight_layout(); plt.savefig('cross_dataset_bangladesh_cm.png', dpi=300, bbox_inches='tight'); plt.show()